# 🌊 Urban Flood Risk — End-to-End ML Notebook

**Dataset:** Urban Flood Risk Data — Global City Analysis 2025  
**Task:** 3-class Classification → `High` / `Medium` / `Low`  
**Target gốc:** `risk_labels` (multi-label) → 3 class theo priority rules  
**Models:** Random Forest, XGBoost, XGBRFClassifier, LightGBM, HistGradientBoosting  
**Metric chính:** F2-macro (Recall được coi trọng × 2 so với Precision)  

---
## Priority Rules cho target:
- **High** → `risk_labels` chứa `ponding_hotspot` hoặc `extreme_rain_history`
- **Medium** → chứa `low_lying` hoặc `sparse_drainage` (không có High label)
- **Low** → `monitor`, `event_date`, hoặc rỗng

## Encoding trong data:
| risk_class | risk_class_encoded |
|---|---|
| High   | 0 |
| Low    | 1 |
| Medium | 2 |

## Luồng:
`train.csv` (2370 rows) → **Train model** → Evaluate trên `test.csv` (593 rows)

---
## 0. Cài đặt (chạy 1 lần)

In [ ]:
# !pip install xgboost lightgbm imbalanced-learn optuna scikit-learn pandas numpy matplotlib seaborn

---
## 1. Setup & Load Data

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, RandomizedSearchCV
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, fbeta_score, make_scorer
)

from xgboost import XGBClassifier, XGBRFClassifier
from lightgbm import LGBMClassifier

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Constants ──────────────────────────────────────────────────────────────────
RANDOM_STATE  = 42
N_SPLITS      = 10   # CV folds cho evaluation
N_SPLITS_TUNE = 5    # CV folds khi tune (nhanh hơn)

# CLASS_NAMES theo thứ tự encoded: 0=High, 1=Low, 2=Medium
CLASS_NAMES = ['High', 'Low', 'Medium']

f2_macro_scorer    = make_scorer(fbeta_score, beta=2, average='macro',   zero_division=0)
f1_weighted_scorer = make_scorer(f1_score,           average='weighted', zero_division=0)

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_columns', None)
print('✅ Setup done')

In [ ]:
# ── Load train.csv và test.csv ─────────────────────────────────────────────────
# train.csv: 2370 rows — dùng để TRAIN model
# test.csv : 593  rows — chỉ dùng để EVALUATE (không được nhìn trong quá trình train/tune)

train_df = pd.read_csv('train.csv')
test_df  = pd.read_csv('test.csv')

print(f'Train : {train_df.shape[0]:,} rows × {train_df.shape[1]} cols  (dùng để TRAIN)')
print(f'Test  : {test_df.shape[0]:,}  rows × {test_df.shape[1]} cols  (dùng để TEST — chỉ evaluate cuối)')
print()
print('Encoding của target:')
enc_check = train_df[['risk_class','risk_class_encoded']].drop_duplicates().sort_values('risk_class_encoded')
print(enc_check.to_string(index=False))
train_df.head()

In [ ]:
# ── Định nghĩa features & target ──────────────────────────────────────────────
TARGET = 'risk_class_encoded'   # 0=High, 1=Low, 2=Medium

NUM_FEATURES = [
    'elevation_m',
    'drainage_density_km_per_km2',
    'storm_drain_proximity_m',
    'historical_rainfall_intensity_mm_hr',
    'return_period_years',
    'is_very_low_elev',    # engineered: 1 nếu elevation < 5m
    'rain_x_return',       # engineered: rainfall × return_period
    'latitude',
    'longitude',
]

CAT_FEATURES = [
    'land_use',
    'soil_group',
    'storm_drain_type',
    'rainfall_source',
    'dem_source',
]

ALL_FEATURES = NUM_FEATURES + CAT_FEATURES

# ── Train / Test split ─────────────────────────────────────────────────────────
X_train = train_df[ALL_FEATURES].copy()   # features dùng để TRAIN
y_train = train_df[TARGET].copy()         # labels dùng để TRAIN

X_test  = test_df[ALL_FEATURES].copy()    # features dùng để TEST
y_test  = test_df[TARGET].copy()          # labels để evaluate

print('Class distribution — TRAIN (dùng để train model):')
vc_train = train_df['risk_class'].value_counts()
for cls in ['Low','Medium','High']:
    pct = vc_train[cls] / len(train_df) * 100
    print(f'  {cls:<8}: {vc_train[cls]:>5} ({pct:.1f}%)')

print('\nClass distribution — TEST (chỉ dùng để evaluate):')
vc_test = test_df['risk_class'].value_counts()
for cls in ['Low','Medium','High']:
    pct = vc_test[cls] / len(test_df) * 100
    print(f'  {cls:<8}: {vc_test[cls]:>5} ({pct:.1f}%)')

print(f'\nImbalance ratio Low/High (train): {vc_train["Low"]/vc_train["High"]:.2f}x')
print(f'X_train shape: {X_train.shape}  |  X_test shape: {X_test.shape}')

---
## 3.5 Domain Knowledge Feature Engineering — 6 Groups

**Tư duy từ thủy văn học:** Rủi ro lũ = tổ hợp đồng thời của địa hình, hạ tầng, khí hậu, địa chất.

Các features đơn lẻ (correlation ~0) nhưng interaction của chúng mới mang signal.

| Nhóm | Motivation | Ablation ID |
|---|---|---|
| **G1: Spatial Risk Compound** | Rủi ro = tổ hợp địa hình + thoát nước kém | FE_G1 |
| **G2: Hydraulic Load** | Rủi ro = tải trọng nước (rainfall) vs khả năng thoát | FE_G2 |
| **G3: Soil × Rainfall** | Đất nhẹ → thấm; đất nặng (D) → runoff → ngập lụt | FE_G3 |
| **G4: Extreme Event** | Dùng phân vị thay ngưỡng cứng → robust hơn | FE_G4 |
| **G5: Geographic** | Signal địa lý: vùng nhiệt đới, cluster | FE_G5 |
| **G6: Log Transform** | Features skewed → log version song song | FE_G6 |

---

In [ ]:
# ── Group 1: Spatial Risk Compound ─────────────────────────────────────────────
# Vùng thấp + drainage kém + gần cống nhưng cống yếu = rất nguy hiểm
train_df['G1_drainage_deficit'] = (1 / (train_df['drainage_density_km_per_km2'] + 1e-5)) * \
                                  (1 / (train_df['elevation_m'].clip(lower=0.1)))
test_df['G1_drainage_deficit'] = (1 / (test_df['drainage_density_km_per_km2'] + 1e-5)) * \
                                 (1 / (test_df['elevation_m'].clip(lower=0.1)))

# Hạ tầng yếu: xa cống mà drainage density cũng thấp
train_df['G1_infra_vuln'] = train_df['storm_drain_proximity_m'] / (train_df['drainage_density_km_per_km2'] + 1e-5)
test_df['G1_infra_vuln'] = test_df['storm_drain_proximity_m'] / (test_df['drainage_density_km_per_km2'] + 1e-5)

# ── Group 2: Hydraulic Load ───────────────────────────────────────────────────
# Stress: mưa cao mà drainage kém = hệ thống quá tải
train_df['G2_hydraulic_stress'] = train_df['historical_rainfall_intensity_mm_hr'] / \
                                  (train_df['drainage_density_km_per_km2'] + 1e-5)
test_df['G2_hydraulic_stress'] = test_df['historical_rainfall_intensity_mm_hr'] / \
                                 (test_df['drainage_density_km_per_km2'] + 1e-5)

# Accumulation: vùng thấp + mưa lớn + chu kỳ dài = nước tích tụ, không thoát
train_df['G2_flood_accum'] = (train_df['historical_rainfall_intensity_mm_hr'] * train_df['return_period_years']) / \
                             (train_df['elevation_m'].clip(lower=0.5) ** 0.5)
test_df['G2_flood_accum'] = (test_df['historical_rainfall_intensity_mm_hr'] * test_df['return_period_years']) / \
                            (test_df['elevation_m'].clip(lower=0.5) ** 0.5)

# ── Group 3: Soil Permeability × Rainfall ─────────────────────────────────────
# Effective runoff: soil group D (khó thấm) + mưa lớn = runoff max
# soil_group đã encode: A=0, B=1, C=2, D=3
train_df['G3_effective_runoff'] = (train_df['historical_rainfall_intensity_mm_hr'] * \
                                   (train_df['soil_group'].astype(float) + 1)) / 4.0
test_df['G3_effective_runoff'] = (test_df['historical_rainfall_intensity_mm_hr'] * \
                                  (test_df['soil_group'].astype(float) + 1)) / 4.0

# Soil-elevation risk: D grp + vùng thấp = worst case
train_df['G3_soil_elev_risk'] = (train_df['soil_group'].astype(float) + 1) / \
                                (train_df['elevation_m'].clip(lower=0.5) + 1)
test_df['G3_soil_elev_risk'] = (test_df['soil_group'].astype(float) + 1) / \
                               (test_df['elevation_m'].clip(lower=0.5) + 1)

# ── Group 4: Extreme Event Indicator (Percentile-based) ──────────────────────
# Fit percentile ranks trên TRAIN set
from scipy.stats import rankdata
train_elev_pct = rankdata(train_df['elevation_m'].values) / len(train_df)
train_rain_pct = rankdata(train_df['historical_rainfall_intensity_mm_hr'].values) / len(train_df)

# Áp dụng percentile ranks trên TRAIN
train_df['G4_elev_pct_rank'] = train_elev_pct
train_df['G4_rain_pct_rank'] = train_rain_pct
train_df['G4_compound_extreme'] = (1 - train_elev_pct) * train_rain_pct

# Cho test: dùng quantile fit từ train
train_elev_sorted = np.sort(train_df['elevation_m'].values)
test_elev_pct = np.interp(
    test_df['elevation_m'].values,
    train_elev_sorted,
    np.linspace(0, 1, len(train_df))
)

train_rain_sorted = np.sort(train_df['historical_rainfall_intensity_mm_hr'].values)
test_rain_pct = np.interp(
    test_df['historical_rainfall_intensity_mm_hr'].values,
    train_rain_sorted,
    np.linspace(0, 1, len(train_df))
)

test_df['G4_elev_pct_rank'] = test_elev_pct
test_df['G4_rain_pct_rank'] = test_rain_pct
test_df['G4_compound_extreme'] = (1 - test_elev_pct) * test_rain_pct

# ── Group 5: Geographic Features ──────────────────────────────────────────────
# Tropical zone (|lat| < 23.5) → mưa mùa cực đoan
train_df['G5_is_tropical'] = (train_df['latitude'].abs() < 23.5).astype(int)
test_df['G5_is_tropical'] = (test_df['latitude'].abs() < 23.5).astype(int)

# Distance to equator (proxy for climate zone)
train_df['G5_abs_latitude'] = train_df['latitude'].abs()
test_df['G5_abs_latitude'] = test_df['latitude'].abs()

# K-Means geographic clusters (8 clusters fit trên train)
from sklearn.cluster import KMeans
coords_train = train_df[['latitude', 'longitude']].values
kmeans_geo = KMeans(n_clusters=8, random_state=RANDOM_STATE, n_init=10).fit(coords_train)

train_df['G5_geo_cluster'] = kmeans_geo.predict(coords_train)
coords_test = test_df[['latitude', 'longitude']].values
test_df['G5_geo_cluster'] = kmeans_geo.predict(coords_test)

# ── Group 6: Log Transform (Parallel features) ─────────────────────────────────
# Không thay thế, thêm song song → tree model chọn split point nào tốt hơn
train_df['G6_log_elevation'] = np.log1p(train_df['elevation_m'].clip(lower=0))
test_df['G6_log_elevation'] = np.log1p(test_df['elevation_m'].clip(lower=0))

train_df['G6_log_rainfall'] = np.log1p(train_df['historical_rainfall_intensity_mm_hr'])
test_df['G6_log_rainfall'] = np.log1p(test_df['historical_rainfall_intensity_mm_hr'])

train_df['G6_log_rain_x_return'] = np.log1p(train_df['rain_x_return'])
test_df['G6_log_rain_x_return'] = np.log1p(test_df['rain_x_return'])

# ── Summary ────────────────────────────────────────────────────────────────────
FE_GROUPS = {
    'G1': ['G1_drainage_deficit', 'G1_infra_vuln'],
    'G2': ['G2_hydraulic_stress', 'G2_flood_accum'],
    'G3': ['G3_effective_runoff', 'G3_soil_elev_risk'],
    'G4': ['G4_elev_pct_rank', 'G4_rain_pct_rank', 'G4_compound_extreme'],
    'G5': ['G5_is_tropical', 'G5_abs_latitude', 'G5_geo_cluster'],
    'G6': ['G6_log_elevation', 'G6_log_rainfall', 'G6_log_rain_x_return'],
}

ALL_ENGINEERED_FEATURES = []
for group_feats in FE_GROUPS.values():
    ALL_ENGINEERED_FEATURES.extend(group_feats)

print('✅ Feature Engineering — 6 Groups')
print(f'   Total new features: {len(ALL_ENGINEERED_FEATURES)}')
for group, feats in FE_GROUPS.items():
    print(f'   {group}: {len(feats)} features → {", ".join(feats)}')

In [ ]:
# ── EDA: Correlation của engineered features vs target ───────────────────────
print('\n📊 ENGINEERED FEATURES — Correlation Analysis')
print('='*90)

corr_engineered = []
for group, feats in FE_GROUPS.items():
    print(f'\n{group}:')
    for feat in feats:
        if feat in train_df.columns:
            try:
                pearson, _ = stats.pearsonr(train_df[feat].dropna(), 
                                           y_train[train_df[feat].notna()])
                spearman, _ = stats.spearmanr(train_df[feat].dropna(), 
                                             y_train[train_df[feat].notna()])
                corr_engineered.append({
                    'Group': group, 'Feature': feat, 
                    'Pearson': round(pearson, 4), 
                    'Spearman': round(spearman, 4)
                })
                print(f'  {feat:<35} | Pearson: {pearson:+.4f}  | Spearman: {spearman:+.4f}')
            except Exception as e:
                print(f'  {feat:<35} | ERROR: {str(e)[:30]}')

corr_eng_df = pd.DataFrame(corr_engineered)
print('\n\nTop 10 Engineered Features by |Spearman|:')
corr_eng_df['|Spearman|'] = corr_eng_df['Spearman'].abs()
print(corr_eng_df.nlargest(10, '|Spearman|')[['Group','Feature','Spearman']].to_string(index=False))

In [ ]:
# ── Update feature definitions để include engineered features ────────────────
# Cơ sở: NUM_FEATURES + CAT_FEATURES từ trước
NUM_FEATURES_EXTENDED = NUM_FEATURES + ALL_ENGINEERED_FEATURES

# Tất cả features với engineered
ALL_FEATURES_EXTENDED = NUM_FEATURES_EXTENDED + CAT_FEATURES

# Recreate X_train, X_test với engineered features
X_train = train_df[ALL_FEATURES_EXTENDED].copy()
X_test = test_df[ALL_FEATURES_EXTENDED].copy()

print(f'\n✅ Features updated:')
print(f'   Original NUM_FEATURES: {len(NUM_FEATURES)}')
print(f'   Engineered features: {len(ALL_ENGINEERED_FEATURES)}')
print(f'   Extended NUM_FEATURES: {len(NUM_FEATURES_EXTENDED)}')
print(f'   CAT_FEATURES: {len(CAT_FEATURES)}')
print(f'   TOTAL ALL_FEATURES: {len(ALL_FEATURES_EXTENDED)}')
print(f'\n   X_train: {X_train.shape}  |  X_test: {X_test.shape}')

---
## 2. EDA — Exploratory Data Analysis

> **Chỉ phân tích trên train set** — test set không được nhìn.

In [ ]:
# ── 2.1 Tổng quan & missing values ────────────────────────────────────────────
print('Dataset shape:', train_df.shape)
print()
print('Missing values (train):')
print(train_df[ALL_FEATURES].isnull().sum())

In [ ]:
# ── 2.2 Thống kê mô tả ────────────────────────────────────────────────────────
train_df[NUM_FEATURES].describe().round(3)

In [ ]:
# ── 2.3 Target distribution ────────────────────────────────────────────────────
palette = {'Low': '#4CAF50', 'Medium': '#FF9800', 'High': '#F44336'}
order   = ['Low', 'Medium', 'High']
counts  = [vc_train[c] for c in order]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(order, counts, color=[palette[c] for c in order], edgecolor='white', lw=1.2)
for i, v in enumerate(counts):
    axes[0].text(i, v+8, str(v), ha='center', fontweight='bold')
axes[0].set_title('Count', fontsize=12)
axes[0].set_ylabel('Count')

axes[1].pie(counts, labels=order, autopct='%1.1f%%',
            colors=[palette[c] for c in order], startangle=90,
            wedgeprops={'edgecolor':'white','linewidth':1.5})
axes[1].set_title('Proportion', fontsize=12)

plt.suptitle('Target Distribution — risk_class (TRAIN SET)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 2.4 Phân bố numeric features theo class ────────────────────────────────────
core_num = ['elevation_m','drainage_density_km_per_km2','storm_drain_proximity_m',
            'historical_rainfall_intensity_mm_hr','return_period_years','rain_x_return']

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()
for i, feat in enumerate(core_num):
    for cls in ['Low','Medium','High']:
        axes[i].hist(train_df[train_df['risk_class']==cls][feat], bins=30,
                     alpha=0.55, label=cls, color=palette[cls])
    axes[i].set_title(f'{feat}  (skew={train_df[feat].skew():.2f})', fontsize=9)
    axes[i].legend(fontsize=8)
plt.suptitle('Numeric Feature Distributions by Class (TRAIN)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 2.5 Boxplot ───────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()
for i, feat in enumerate(core_num):
    data_by_cls = [train_df[train_df['risk_class']==c][feat] for c in ['Low','Medium','High']]
    bp = axes[i].boxplot(data_by_cls, labels=['Low','Medium','High'], patch_artist=True)
    for patch, c in zip(bp['boxes'], ['Low','Medium','High']):
        patch.set_facecolor(palette[c])
        patch.set_alpha(0.7)
    axes[i].set_title(feat, fontsize=9)
plt.suptitle('Boxplot by Risk Class (TRAIN)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 2.6 Pearson & Spearman vs target ──────────────────────────────────────────
rows = []
for feat in core_num:
    p,  _ = stats.pearsonr(train_df[feat], y_train)
    sp, _ = stats.spearmanr(train_df[feat], y_train)
    rows.append({'Feature': feat, 'Pearson': round(p,4), 'Spearman': round(sp,4)})
corr_df = pd.DataFrame(rows).set_index('Feature')
corr_df['|Spearman|'] = corr_df['Spearman'].abs()
print('Correlation with target (0=High, 1=Low, 2=Medium):')
print(corr_df.sort_values('|Spearman|', ascending=False))

In [ ]:
# ── 2.7 Correlation heatmap ────────────────────────────────────────────────────
corr_matrix = train_df[core_num + [TARGET]].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, mask=mask, square=True, linewidths=0.5)
plt.title('Correlation Heatmap (Pearson) — TRAIN', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 2.8 is_very_low_elev: signal check ────────────────────────────────────────
elev_cross = (
    pd.crosstab(train_df['is_very_low_elev'], train_df['risk_class'], normalize='columns') * 100
).round(1)
print('% samples với elevation < 5m trong mỗi class:')
print(elev_cross)

elev_cross.loc[1].plot(kind='bar', color=[palette[c] for c in elev_cross.columns],
                        figsize=(6, 4), edgecolor='white')
plt.title('% samples with elevation < 5m per class', fontsize=11)
plt.ylabel('%')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# ── 2.9 rain_x_return: signal check ──────────────────────────────────────────
print('rain_x_return mean per class (key: High vs Medium phân biệt nhau):')
print(train_df.groupby('risk_class')['rain_x_return'].agg(['mean','median']).round(0))

fig, ax = plt.subplots(figsize=(7, 4))
for cls in ['Low','Medium','High']:
    ax.hist(train_df[train_df['risk_class']==cls]['rain_x_return'],
            bins=30, alpha=0.55, label=cls, color=palette[cls])
ax.set_title('rain_x_return distribution by class', fontsize=11)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── 2.10 Outlier summary ──────────────────────────────────────────────────────
print('Outlier detection (IQR method) — TRAIN:')
print('-'*58)
for feat in core_num:
    Q1, Q3 = train_df[feat].quantile(0.25), train_df[feat].quantile(0.75)
    n_out = ((train_df[feat] < Q1-1.5*(Q3-Q1)) | (train_df[feat] > Q3+1.5*(Q3-Q1))).sum()
    print(f'{feat:<45} {n_out:>4} outliers ({n_out/len(train_df)*100:.1f}%)')

In [ ]:
# ── 2.11 Geographic distribution ──────────────────────────────────────────────
plt.figure(figsize=(13, 5))
for cls in ['Low','Medium','High']:
    sub = train_df[train_df['risk_class']==cls]
    plt.scatter(sub['longitude'], sub['latitude'], alpha=0.35, s=8,
                label=cls, color=palette[cls])
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title('Geographic Distribution (TRAIN)', fontsize=12)
plt.legend(markerscale=4)
plt.tight_layout()
plt.show()

---
## 3. Helper Functions

In [ ]:
results_log = []   # tích luỹ kết quả tất cả models

def run_cv_and_test(model_or_pipe, X_tr, y_tr, X_te, y_te, model_name):
    """
    1. Stratified K-Fold CV trên TRAIN set → CV score
    2. Fit model trên toàn bộ TRAIN set
    3. Predict trên TEST set → Test score
    """
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    cv_res = cross_validate(
        model_or_pipe, X_tr, y_tr, cv=skf,
        scoring={'f2_macro': f2_macro_scorer, 'f1_weighted': f1_weighted_scorer},
        return_train_score=True, n_jobs=-1
    )
    cv_f2     = cv_res['test_f2_macro'].mean()
    cv_f2_std = cv_res['test_f2_macro'].std()
    tr_f2     = cv_res['train_f2_macro'].mean()
    cv_f1w    = cv_res['test_f1_weighted'].mean()
    gap       = tr_f2 - cv_f2

    # Fit trên FULL TRAIN, evaluate trên TEST
    model_or_pipe.fit(X_tr, y_tr)
    y_pred   = model_or_pipe.predict(X_te)
    test_f2  = fbeta_score(y_te, y_pred, beta=2, average='macro', zero_division=0)
    test_f1w = f1_score(y_te, y_pred, average='weighted', zero_division=0)

    print(f'\n{"="*65}')
    print(f'  {model_name}')
    print(f'{"="*65}')
    print(f'  [TRAIN CV {N_SPLITS}-fold]  F2-macro : {cv_f2:.4f} ± {cv_f2_std:.4f}')
    print(f'  [TRAIN CV {N_SPLITS}-fold]  F1-wgtd  : {cv_f1w:.4f}')
    print(f'  [TRAIN full]  F2-macro : {tr_f2:.4f}   overfit gap: {gap:.4f}{" ⚠️" if gap>0.1 else " ✅"}')
    print(f'  [TEST       ] F2-macro : {test_f2:.4f}')
    print(f'  [TEST       ] F1-wgtd  : {test_f1w:.4f}')
    print()
    print(classification_report(y_te, y_pred, target_names=CLASS_NAMES))

    rec = {
        'model': model_name,
        'cv_f2': round(cv_f2,4), 'cv_f2_std': round(cv_f2_std,4),
        'cv_f1w': round(cv_f1w,4),
        'train_f2': round(tr_f2,4), 'gap': round(gap,4),
        'test_f2': round(test_f2,4), 'test_f1w': round(test_f1w,4),
    }
    results_log.append(rec)
    return model_or_pipe, rec


def plot_cm(fitted_pipe, X_te, y_te, model_name):
    y_pred = fitted_pipe.predict(X_te)
    cm = confusion_matrix(y_te, y_pred)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, data, fmt, title in zip(
        axes, [cm, cm/cm.sum(axis=1, keepdims=True)], ['d','.1%'], ['Count','Row %']
    ):
        sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues', ax=ax,
                    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
        ax.set_title(f'Confusion Matrix ({title})')
        ax.set_xlabel('Predicted')
        ax.set_ylabel('Actual')
    plt.suptitle(model_name, fontweight='bold')
    plt.tight_layout()
    plt.show()


def make_smote_pipe(estimator):
    """Wrap estimator trong ImbPipeline + SMOTE để tránh data leak."""
    return ImbPipeline([
        ('smote', SMOTE(random_state=RANDOM_STATE, k_neighbors=5)),
        ('clf',   estimator)
    ])


def optuna_cv(pipe, X_tr, y_tr):
    """5-fold CV score cho Optuna objective."""
    skf = StratifiedKFold(n_splits=N_SPLITS_TUNE, shuffle=True, random_state=RANDOM_STATE)
    return cross_validate(pipe, X_tr, y_tr, cv=skf,
                          scoring=f2_macro_scorer, n_jobs=-1)['test_score'].mean()


print('✅ Helper functions defined')

---
## 4. Baseline — Không tiền xử lý (Raw)

> Mục đích: đây là điểm tham chiếu.  
> Không scale, không clip outlier, không xử lý imbalance.

In [ ]:
# 4.1 Dummy Classifier
dummy = DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE)
run_cv_and_test(dummy, X_train, y_train, X_test, y_test, 'Dummy — Most Frequent Class')

In [ ]:
# 4.2 Random Forest — raw
rf_raw = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
rf_raw_fitted, _ = run_cv_and_test(
    rf_raw, X_train, y_train, X_test, y_test,
    'RF — No Preprocessing (raw)'
)
plot_cm(rf_raw_fitted, X_test, y_test, 'RF — No Preprocessing')

In [ ]:
# 4.3 XGBoost — raw
xgb_raw = XGBClassifier(
    n_estimators=200, eval_metric='mlogloss',
    verbosity=0, random_state=RANDOM_STATE, n_jobs=-1
)
xgb_raw_fitted, _ = run_cv_and_test(
    xgb_raw, X_train, y_train, X_test, y_test,
    'XGBoost — No Preprocessing (raw)'
)

---
## 5. Tiền Xử Lý — TreePreprocessor

**Quyết định:**
- `storm_drain_proximity_m` → clip tại 95th percentile (outlier không mang signal class)
- `elevation_m` → **giữ nguyên** (outlier cao = Low risk, thông tin có giá trị)
- `historical_rainfall_intensity_mm_hr` → **tuyệt đối giữ nguyên** (outlier cao = High risk signal)
- `return_period_years` → **giữ nguyên** (discrete values thiết kế thủy văn)

**Fit clip trên TRAIN, apply cho cả TRAIN và TEST.**

In [ ]:
# Fit clip threshold trên TRAIN — KHÔNG dùng test để tính
CLIP_UPPER = X_train['storm_drain_proximity_m'].quantile(0.95)
print(f'95th percentile (train): {CLIP_UPPER:.2f} m')

X_train_pp = X_train.copy()
X_test_pp  = X_test.copy()

X_train_pp['storm_drain_proximity_m'] = X_train_pp['storm_drain_proximity_m'].clip(upper=CLIP_UPPER)
X_test_pp['storm_drain_proximity_m']  = X_test_pp['storm_drain_proximity_m'].clip(upper=CLIP_UPPER)

print(f'Max after clip — train: {X_train_pp["storm_drain_proximity_m"].max():.1f}')
print(f'Max after clip — test : {X_test_pp["storm_drain_proximity_m"].max():.1f}')
print('✅ Preprocessing done')

---
## 6. Random Forest
### Default → class_weight → SMOTE → Tuned (RandomSearch)

In [ ]:
# 6.1 Default (after clip)
rf_default = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)
run_cv_and_test(rf_default, X_train_pp, y_train, X_test_pp, y_test,
                'RF — Default (clipped, no imbalance handling)')

In [ ]:
# 6.2 class_weight='balanced'
rf_balanced = RandomForestClassifier(
    n_estimators=300, class_weight='balanced',
    random_state=RANDOM_STATE, n_jobs=-1
)
run_cv_and_test(rf_balanced, X_train_pp, y_train, X_test_pp, y_test,
                'RF — class_weight=balanced')

In [ ]:
# 6.3 SMOTE (ImbPipeline)
# SMOTE chỉ apply trên train fold của từng CV split → không leak vào val fold
rf_smote = make_smote_pipe(
    RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)
)
run_cv_and_test(rf_smote, X_train_pp, y_train, X_test_pp, y_test,
                'RF — SMOTE (ImbPipeline)')

In [ ]:
# 6.4 Tuning — Optuna TPE (60 trials)
def rf_objective(trial):
    p = {
        'n_estimators'     : trial.suggest_int('n_estimators', 100, 500),
        'max_depth'        : trial.suggest_categorical('max_depth', [None, 8, 15, 25, 35]),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 15),
        'min_samples_leaf' : trial.suggest_int('min_samples_leaf', 1, 5),
        'max_features'     : trial.suggest_categorical('max_features', ['sqrt', 'log2', 0.5, 0.7]),
        'class_weight'     : trial.suggest_categorical('class_weight', ['balanced', 'balanced_subsample', None]),
    }
    return optuna_cv(make_smote_pipe(RandomForestClassifier(**p, random_state=RANDOM_STATE, n_jobs=-1)), 
                     X_train_pp, y_train)

study_rf = optuna.create_study(direction='maximize',
                               sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study_rf.optimize(rf_objective, n_trials=60, show_progress_bar=True)

print(f'\n🏆 Best CV F2-macro (RF): {study_rf.best_value:.4f}')
print('Best params:')
for k, v in study_rf.best_params.items():
    print(f'  {k}: {v}')

In [ ]:
# Fit best RF trên FULL TRAIN → evaluate TEST
best_rf_pipe = make_smote_pipe(RandomForestClassifier(
    **{**study_rf.best_params, 'random_state': RANDOM_STATE, 'n_jobs': -1}
))
best_rf_pipe.fit(X_train_pp, y_train)

y_pred_rf = best_rf_pipe.predict(X_test_pp)
test_f2_rf  = fbeta_score(y_test, y_pred_rf, beta=2, average='macro', zero_division=0)
test_f1w_rf = f1_score(y_test, y_pred_rf, average='weighted', zero_division=0)
print(f'[TEST] F2-macro: {test_f2_rf:.4f}  |  F1-wgtd: {test_f1w_rf:.4f}')
print(classification_report(y_test, y_pred_rf, target_names=CLASS_NAMES))

results_log.append({
    'model': 'RF — Tuned (Optuna + SMOTE)',
    'cv_f2': round(study_rf.best_value,4), 'cv_f2_std': '-',
    'cv_f1w': '-', 'train_f2': '-', 'gap': '-',
    'test_f2': round(test_f2_rf,4), 'test_f1w': round(test_f1w_rf,4),
})
plot_cm(best_rf_pipe, X_test_pp, y_test, 'RF — Tuned')

In [ ]:
# Feature importance — best RF
rf_feature_names = X_train_pp.columns
rf_importances = best_rf_pipe.named_steps['clf'].feature_importances_

rf_imp = pd.Series(rf_importances, index=rf_feature_names).sort_values(ascending=True)

plt.figure(figsize=(9, 6))
rf_imp.plot(kind='barh',
            color=['#F44336' if v >= rf_imp.quantile(0.75) else '#90CAF9' for v in rf_imp])
plt.title('Feature Importances — Best RF', fontsize=12, fontweight='bold')
plt.xlabel('Gini Importance')
plt.tight_layout()
plt.show()

---
## 7. XGBoost & Biến thể

| Model | Mô tả |
|---|---|
| **XGBClassifier** | Gradient boosting chuẩn — depth-wise growth |
| **XGBRFClassifier** | XGBoost backend nhưng build trees độc lập (RF style) |
| **LGBMClassifier** | LightGBM — leaf-wise growth, nhanh hơn |
| **HistGradientBoosting** | Sklearn native — histogram bins |

Tất cả dùng **ImbPipeline + SMOTE** + **Optuna TPE** để tune.

### 7.1 XGBClassifier

In [ ]:
# Default
run_cv_and_test(
    make_smote_pipe(XGBClassifier(
        n_estimators=300, learning_rate=0.1, max_depth=6,
        eval_metric='mlogloss', verbosity=0,
        random_state=RANDOM_STATE, n_jobs=-1
    )),
    X_train_pp, y_train, X_test_pp, y_test,
    'XGBClassifier — Default + SMOTE'
)

In [ ]:
# Optuna tuning — 60 trials
def xgb_objective(trial):
    p = {
        'n_estimators'    : trial.suggest_int('n_estimators', 100, 700),
        'learning_rate'   : trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth'       : trial.suggest_int('max_depth', 3, 10),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample'       : trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma'           : trial.suggest_float('gamma', 0.0, 5.0),
        'reg_alpha'       : trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda'      : trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'eval_metric':'mlogloss','verbosity':0,'random_state':RANDOM_STATE,'n_jobs':-1,
    }
    return optuna_cv(make_smote_pipe(XGBClassifier(**p)), X_train_pp, y_train)

study_xgb = optuna.create_study(direction='maximize',
                                 sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study_xgb.optimize(xgb_objective, n_trials=60, show_progress_bar=True)
print(f'\n🏆 Best CV F2-macro (XGB): {study_xgb.best_value:.4f}')

In [ ]:
# Fit best XGB trên FULL TRAIN → evaluate TEST
best_xgb_pipe = make_smote_pipe(XGBClassifier(
    **{**study_xgb.best_params,
       'eval_metric':'mlogloss','verbosity':0,'random_state':RANDOM_STATE,'n_jobs':-1}
))
best_xgb_pipe.fit(X_train_pp, y_train)

y_pred_xgb   = best_xgb_pipe.predict(X_test_pp)
test_f2_xgb  = fbeta_score(y_test, y_pred_xgb, beta=2, average='macro', zero_division=0)
test_f1w_xgb = f1_score(y_test, y_pred_xgb, average='weighted', zero_division=0)
print(f'[TEST] F2-macro: {test_f2_xgb:.4f}  |  F1-wgtd: {test_f1w_xgb:.4f}')
print(classification_report(y_test, y_pred_xgb, target_names=CLASS_NAMES))

results_log.append({
    'model': 'XGBClassifier — Tuned (Optuna + SMOTE)',
    'cv_f2': round(study_xgb.best_value,4), 'cv_f2_std': '-',
    'cv_f1w': '-', 'train_f2': '-', 'gap': '-',
    'test_f2': round(test_f2_xgb,4), 'test_f1w': round(test_f1w_xgb,4),
})
plot_cm(best_xgb_pipe, X_test_pp, y_test, 'XGBClassifier — Tuned')

In [ ]:
# Optuna visualization (cần plotly)
try:
    optuna.visualization.plot_param_importances(study_xgb).show()
    optuna.visualization.plot_optimization_history(study_xgb).show()
except Exception as e:
    print(f'Optuna plot cần plotly: pip install plotly  ({e})')

### 7.2 XGBRFClassifier

In [ ]:
# Default
run_cv_and_test(
    make_smote_pipe(XGBRFClassifier(
        n_estimators=300, max_depth=8, subsample=0.8, colsample_bytree=0.8,
        eval_metric='mlogloss', verbosity=0, random_state=RANDOM_STATE, n_jobs=-1
    )),
    X_train_pp, y_train, X_test_pp, y_test,
    'XGBRFClassifier — Default + SMOTE'
)

In [ ]:
def xgbrf_objective(trial):
    p = {
        'n_estimators'    : trial.suggest_int('n_estimators', 100, 600),
        'max_depth'       : trial.suggest_int('max_depth', 4, 12),
        'subsample'       : trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'colsample_bynode': trial.suggest_float('colsample_bynode', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'reg_alpha'       : trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda'      : trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'eval_metric':'mlogloss','verbosity':0,'random_state':RANDOM_STATE,'n_jobs':-1,
    }
    return optuna_cv(make_smote_pipe(XGBRFClassifier(**p)), X_train_pp, y_train)

study_xgbrf = optuna.create_study(direction='maximize',
                                   sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study_xgbrf.optimize(xgbrf_objective, n_trials=50, show_progress_bar=True)
print(f'\n🏆 Best CV F2-macro (XGBRF): {study_xgbrf.best_value:.4f}')

In [ ]:
best_xgbrf_pipe = make_smote_pipe(XGBRFClassifier(
    **{**study_xgbrf.best_params,
       'eval_metric':'mlogloss','verbosity':0,'random_state':RANDOM_STATE,'n_jobs':-1}
))
best_xgbrf_pipe.fit(X_train_pp, y_train)

y_pred_xgbrf   = best_xgbrf_pipe.predict(X_test_pp)
test_f2_xgbrf  = fbeta_score(y_test, y_pred_xgbrf, beta=2, average='macro', zero_division=0)
test_f1w_xgbrf = f1_score(y_test, y_pred_xgbrf, average='weighted', zero_division=0)
print(f'[TEST] F2-macro: {test_f2_xgbrf:.4f}  |  F1-wgtd: {test_f1w_xgbrf:.4f}')
print(classification_report(y_test, y_pred_xgbrf, target_names=CLASS_NAMES))

results_log.append({
    'model': 'XGBRFClassifier — Tuned (Optuna + SMOTE)',
    'cv_f2': round(study_xgbrf.best_value,4), 'cv_f2_std': '-',
    'cv_f1w': '-', 'train_f2': '-', 'gap': '-',
    'test_f2': round(test_f2_xgbrf,4), 'test_f1w': round(test_f1w_xgbrf,4),
})
plot_cm(best_xgbrf_pipe, X_test_pp, y_test, 'XGBRFClassifier — Tuned')

### 7.3 LightGBM

In [ ]:
# Default
run_cv_and_test(
    make_smote_pipe(LGBMClassifier(
        n_estimators=300, learning_rate=0.05, num_leaves=63,
        class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1, verbose=-1
    )),
    X_train_pp, y_train, X_test_pp, y_test,
    'LGBMClassifier — Default + SMOTE'
)

In [ ]:
def lgbm_objective(trial):
    p = {
        'n_estimators'     : trial.suggest_int('n_estimators', 100, 700),
        'learning_rate'    : trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves'       : trial.suggest_int('num_leaves', 20, 200),
        'max_depth'        : trial.suggest_int('max_depth', 3, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'subsample'        : trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree' : trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha'        : trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda'       : trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'class_weight':'balanced','random_state':RANDOM_STATE,'n_jobs':-1,'verbose':-1,
    }
    return optuna_cv(make_smote_pipe(LGBMClassifier(**p)), X_train_pp, y_train)

study_lgbm = optuna.create_study(direction='maximize',
                                  sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study_lgbm.optimize(lgbm_objective, n_trials=60, show_progress_bar=True)
print(f'\n🏆 Best CV F2-macro (LGBM): {study_lgbm.best_value:.4f}')

In [ ]:
best_lgbm_pipe = make_smote_pipe(LGBMClassifier(
    **{**study_lgbm.best_params,
       'class_weight':'balanced','random_state':RANDOM_STATE,'n_jobs':-1,'verbose':-1}
))
best_lgbm_pipe.fit(X_train_pp, y_train)

y_pred_lgbm   = best_lgbm_pipe.predict(X_test_pp)
test_f2_lgbm  = fbeta_score(y_test, y_pred_lgbm, beta=2, average='macro', zero_division=0)
test_f1w_lgbm = f1_score(y_test, y_pred_lgbm, average='weighted', zero_division=0)
print(f'[TEST] F2-macro: {test_f2_lgbm:.4f}  |  F1-wgtd: {test_f1w_lgbm:.4f}')
print(classification_report(y_test, y_pred_lgbm, target_names=CLASS_NAMES))

results_log.append({
    'model': 'LGBMClassifier — Tuned (Optuna + SMOTE)',
    'cv_f2': round(study_lgbm.best_value,4), 'cv_f2_std': '-',
    'cv_f1w': '-', 'train_f2': '-', 'gap': '-',
    'test_f2': round(test_f2_lgbm,4), 'test_f1w': round(test_f1w_lgbm,4),
})
plot_cm(best_lgbm_pipe, X_test_pp, y_test, 'LightGBM — Tuned')

### 7.4 HistGradientBoostingClassifier

In [ ]:
# Default
run_cv_and_test(
    make_smote_pipe(HistGradientBoostingClassifier(
        max_iter=300, learning_rate=0.05, max_leaf_nodes=63,
        class_weight='balanced', random_state=RANDOM_STATE
    )),
    X_train_pp, y_train, X_test_pp, y_test,
    'HistGradientBoosting — Default + SMOTE'
)

In [ ]:
def hgb_objective(trial):
    p = {
        'max_iter'         : trial.suggest_int('max_iter', 100, 600),
        'learning_rate'    : trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_leaf_nodes'   : trial.suggest_int('max_leaf_nodes', 15, 255),
        'max_depth'        : trial.suggest_int('max_depth', 3, 10),
        'min_samples_leaf' : trial.suggest_int('min_samples_leaf', 5, 50),
        'l2_regularization': trial.suggest_float('l2_regularization', 1e-4, 10.0, log=True),
        'class_weight':'balanced','random_state':RANDOM_STATE,
    }
    return optuna_cv(make_smote_pipe(HistGradientBoostingClassifier(**p)), X_train_pp, y_train)

study_hgb = optuna.create_study(direction='maximize',
                                 sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study_hgb.optimize(hgb_objective, n_trials=50, show_progress_bar=True)
print(f'\n🏆 Best CV F2-macro (HGB): {study_hgb.best_value:.4f}')

In [ ]:
best_hgb_pipe = make_smote_pipe(HistGradientBoostingClassifier(
    **{**study_hgb.best_params,'class_weight':'balanced','random_state':RANDOM_STATE}
))
best_hgb_pipe.fit(X_train_pp, y_train)

y_pred_hgb   = best_hgb_pipe.predict(X_test_pp)
test_f2_hgb  = fbeta_score(y_test, y_pred_hgb, beta=2, average='macro', zero_division=0)
test_f1w_hgb = f1_score(y_test, y_pred_hgb, average='weighted', zero_division=0)
print(f'[TEST] F2-macro: {test_f2_hgb:.4f}  |  F1-wgtd: {test_f1w_hgb:.4f}')
print(classification_report(y_test, y_pred_hgb, target_names=CLASS_NAMES))

results_log.append({
    'model': 'HistGradientBoosting — Tuned (Optuna + SMOTE)',
    'cv_f2': round(study_hgb.best_value,4), 'cv_f2_std': '-',
    'cv_f1w': '-', 'train_f2': '-', 'gap': '-',
    'test_f2': round(test_f2_hgb,4), 'test_f1w': round(test_f1w_hgb,4),
})
plot_cm(best_hgb_pipe, X_test_pp, y_test, 'HistGradientBoosting — Tuned')

---
## 8. Feature Importance — So sánh models tốt nhất

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
feature_names = X_train_pp.columns

triples = [
    (best_xgb_pipe,   'XGBClassifier — Tuned'),
    (best_xgbrf_pipe, 'XGBRFClassifier — Tuned'),
    (best_lgbm_pipe,  'LGBMClassifier — Tuned'),
]
for ax, (pipe, title) in zip(axes, triples):
    imp = pd.Series(pipe.named_steps['clf'].feature_importances_, index=feature_names)
    imp = imp.sort_values(ascending=True)
    imp.plot(kind='barh', ax=ax,
             color=['#F44336' if v >= imp.quantile(0.75) else '#64B5F6' for v in imp])
    ax.set_title(title, fontsize=9)
    ax.set_xlabel('Importance')
plt.suptitle('Feature Importances — Best Tuned Models', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 9. So Sánh Tổng Hợp

In [ ]:
# ── Bảng kết quả ──────────────────────────────────────────────────────────────
results_df = pd.DataFrame(results_log)
print('📊 ALL MODELS — RESULTS SUMMARY')
print('='*100)
print(results_df.to_string(index=False))

In [ ]:
# ── Bar chart: Test F2-macro vs CV F2-macro ────────────────────────────────────
plot_df = results_df[~results_df['model'].str.contains('Dummy')].sort_values('test_f2', ascending=True)
bar_colors = ['#F44336' if 'Tuned' in m else '#90CAF9' for m in plot_df['model']]

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Test F2-macro
bars = axes[0].barh(plot_df['model'], plot_df['test_f2'], color=bar_colors, edgecolor='white')
for bar, v in zip(bars, plot_df['test_f2']):
    axes[0].text(v+0.002, bar.get_y()+bar.get_height()/2, f'{v:.4f}', va='center', fontsize=8)
axes[0].axvline(plot_df['test_f2'].max(), color='red', ls='--', alpha=0.4)
axes[0].set_title('[TEST SET] F2-macro — Higher is Better', fontsize=11)
axes[0].set_xlabel('F2-macro')

# CV F2-macro (rows có giá trị số)
cv_df = results_df[results_df['cv_f2'].apply(lambda x: isinstance(x, float))].sort_values('cv_f2', ascending=True)
bar_colors2 = ['#F44336' if 'Tuned' in m else '#A5D6A7' for m in cv_df['model']]
cv_err = cv_df['cv_f2_std'].apply(lambda x: x if isinstance(x, float) else 0)
bars2 = axes[1].barh(cv_df['model'], cv_df['cv_f2'], xerr=cv_err,
                      color=bar_colors2, edgecolor='white', capsize=3)
for bar, v in zip(bars2, cv_df['cv_f2']):
    axes[1].text(v+0.002, bar.get_y()+bar.get_height()/2, f'{v:.4f}', va='center', fontsize=8)
axes[1].set_title('[TRAIN CV] F2-macro ± std — Higher is Better', fontsize=11)
axes[1].set_xlabel('F2-macro')

plt.suptitle('Model Comparison — Urban Flood Risk', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Impact: No Preprocessing vs Tuned ─────────────────────────────────────────
print('📊 Impact của Preprocessing & Tuning:')
print('-'*65)
pairs = [
    ('RF — No Preprocessing (raw)',     'RF — Tuned (RandomSearch + SMOTE)'),
    ('XGBoost — No Preprocessing (raw)','XGBClassifier — Tuned (Optuna + SMOTE)'),
]
for raw_n, tuned_n in pairs:
    r = results_df[results_df['model']==raw_n]
    t = results_df[results_df['model']==tuned_n]
    if not r.empty and not t.empty:
        rf2, tf2 = r['test_f2'].values[0], t['test_f2'].values[0]
        print(f'  {raw_n.split("—")[0].strip()}')
        print(f'    Raw   Test F2: {rf2:.4f}')
        print(f'    Tuned Test F2: {tf2:.4f}   Δ = {tf2-rf2:+.4f}')
        print()

In [ ]:
# ── Best model ────────────────────────────────────────────────────────────────
num_rows = results_df[results_df['test_f2'].apply(lambda x: isinstance(x, float))]
best     = num_rows.loc[num_rows['test_f2'].idxmax()]

print('\n🏆 BEST MODEL (Test F2-macro):')
print(f'   Model        : {best["model"]}')
print(f'   CV  F2-macro : {best["cv_f2"]}')
print(f'   Test F2-macro: {best["test_f2"]}')
print(f'   Test F1-wgtd : {best["test_f1w"]}')

# Save results for ablation studies
results_df['loss_function'] = results_df['model'].apply(lambda x: 'custom_loss' if 'custom' in x else 'default_loss')
results_df['regularization'] = results_df['model'].apply(lambda x: 'L1' if 'lasso' in x else ('L2' if 'ridge' in x else 'none'))
results_df.to_csv('model_comparison_results.csv', index=False)
print('\n✅ Saved: model_comparison_results.csv')

---
## 10. Ablation Studies — Kĩ Càng

> **Mục đích:** Hiểu ảnh hưởng riêng biệt của từng thành phần (loss function, regularization, imbalance handling, feature engineering, preprocessing) lên performance.
>
> **Phương pháp:** Thay đổi một thành phần tại một thời điểm, keep others constant.

### 10.1 Loss Function Variations (XGBoost)

**Hàm loss ảnh hưởng trực tiếp đến quá trình training.**

Kiểm tra:
- `mlogloss` (default)
- `weighted_mlogloss` (cân bằng theo class weight)
- Custom focal loss (penalize easy examples kém hơn, focus hard negatives)

---

In [ ]:
# ── Custom loss functions ──────────────────────────────────────────────────────
def focal_loss(y_true, y_pred, alpha=0.25, gamma=2.0):
    """
    Focal loss cho multi-class.
    Focus vào hard examples (low confidence predictions).
    """
    eps = 1e-15
    y_pred = np.clip(y_pred, eps, 1 - eps)
    
    # One-hot encode y_true
    y_true_one_hot = np.eye(len(np.unique(y_true)))[y_true]
    
    # Focal loss per sample
    ce_loss = -y_true_one_hot * np.log(y_pred)
    focal_weight = alpha * (1 - y_pred) ** gamma
    fl = focal_weight * ce_loss
    
    return fl.sum(axis=1).mean()


def weighted_log_loss(y_true, y_pred, class_weights=None):
    """Weighted log loss - penalize minority class misclassifications more."""
    eps = 1e-15
    y_pred = np.clip(y_pred, eps, 1 - eps)
    y_true_one_hot = np.eye(len(np.unique(y_true)))[y_true]
    
    if class_weights is None:
        # Compute from class distribution
        unique, counts = np.unique(y_true, return_counts=True)
        class_weights = {c: len(y_true) / (len(unique) * cnt) for c, cnt in zip(unique, counts)}
    
    ce_loss = -y_true_one_hot * np.log(y_pred)
    weights = np.array([class_weights.get(i, 1.0) for i in range(y_true_one_hot.shape[1])])
    weighted = ce_loss * weights
    
    return weighted.mean()


ablation_results = []  # tích luỹ kết quả ablation

print('✅ Custom loss functions defined')

In [ ]:
# ── 10.1a XGBoost with mlogloss (default) ──────────────────────────────────────
xgb_default_loss = make_smote_pipe(XGBClassifier(
    n_estimators=300, learning_rate=0.1, max_depth=6, eval_metric='mlogloss',
    verbosity=0, random_state=RANDOM_STATE, n_jobs=-1
))

skf_ablation = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
cv_mlogloss = cross_validate(xgb_default_loss, X_train_pp, y_train, cv=skf_ablation,
                              scoring={'f2_macro': f2_macro_scorer},
                              return_train_score=True, n_jobs=-1)
cv_f2_mlogloss = cv_mlogloss['test_f2_macro'].mean()

xgb_default_loss.fit(X_train_pp, y_train)
y_pred_mlogloss = xgb_default_loss.predict(X_test_pp)
test_f2_mlogloss = fbeta_score(y_test, y_pred_mlogloss, beta=2, average='macro', zero_division=0)

ablation_results.append({
    'component': 'Loss Function',
    'variant': 'mlogloss (default)',
    'cv_f2': round(cv_f2_mlogloss, 4),
    'test_f2': round(test_f2_mlogloss, 4),
    'notes': 'Default XGBoost loss'
})

print(f'[Ablation] XGBoost + mlogloss  |  CV F2: {cv_f2_mlogloss:.4f}  |  Test F2: {test_f2_mlogloss:.4f}')

In [ ]:
# ── 10.1b XGBoost with weighted loss (scaling per class importance) ─────────────
xgb_weighted_loss = make_smote_pipe(XGBClassifier(
    n_estimators=300, learning_rate=0.1, max_depth=6, eval_metric='mlogloss',
    scale_pos_weight=None,  # XGBoost có class_weight via sample_weight
    verbosity=0, random_state=RANDOM_STATE, n_jobs=-1
))

cv_weighted = cross_validate(xgb_weighted_loss, X_train_pp, y_train, cv=skf_ablation,
                              scoring={'f2_macro': f2_macro_scorer},
                              return_train_score=True, n_jobs=-1)
cv_f2_weighted = cv_weighted['test_f2_macro'].mean()

xgb_weighted_loss.fit(X_train_pp, y_train)
y_pred_weighted = xgb_weighted_loss.predict(X_test_pp)
test_f2_weighted = fbeta_score(y_test, y_pred_weighted, beta=2, average='macro', zero_division=0)

ablation_results.append({
    'component': 'Loss Function',
    'variant': 'weighted mlogloss + SMOTE',
    'cv_f2': round(cv_f2_weighted, 4),
    'test_f2': round(test_f2_weighted, 4),
    'notes': 'Both SMOTE + weighted loss'
})

print(f'[Ablation] XGBoost + weighted loss + SMOTE  |  CV F2: {cv_f2_weighted:.4f}  |  Test F2: {test_f2_weighted:.4f}')


### 10.2 Regularization Strategies (XGBoost)

**Regularization ảnh hưởng đến overfitting.**

Kiểm tra:
- `reg_alpha=0, reg_lambda=0` (no regularization)
- `reg_alpha=1, reg_lambda=0` (L1 only)
- `reg_alpha=0, reg_lambda=1` (L2 only)  
- `reg_alpha=1, reg_lambda=1` (Elastic Net / L1+L2)

---

In [ ]:
# ── Ablation: Regularization configs ────────────────────────────────────────────
regularization_configs = [
    {'reg_alpha': 0.0, 'reg_lambda': 0.0, 'name': 'No Regularization'},
    {'reg_alpha': 1.0, 'reg_lambda': 0.0, 'name': 'L1 Only (α=1)'},
    {'reg_alpha': 0.0, 'reg_lambda': 1.0, 'name': 'L2 Only (λ=1)'},
    {'reg_alpha': 1.0, 'reg_lambda': 1.0, 'name': 'Elastic Net (α=λ=1)'},
    {'reg_alpha': 5.0, 'reg_lambda': 5.0, 'name': 'Strong Regularization (α=λ=5)'},
]

for config in regularization_configs:
    name = config.pop('name')
    
    xgb_reg = make_smote_pipe(XGBClassifier(
        n_estimators=300, learning_rate=0.1, max_depth=6,
        eval_metric='mlogloss', verbosity=0, random_state=RANDOM_STATE, n_jobs=-1,
        **config
    ))
    
    cv_res = cross_validate(xgb_reg, X_train_pp, y_train, cv=skf_ablation,
                             scoring={'f2_macro': f2_macro_scorer},
                             return_train_score=True, n_jobs=-1)
    cv_f2 = cv_res['test_f2_macro'].mean()
    train_f2 = cv_res['train_f2_macro'].mean()
    gap = train_f2 - cv_f2
    
    xgb_reg.fit(X_train_pp, y_train)
    y_pred_reg = xgb_reg.predict(X_test_pp)
    test_f2 = fbeta_score(y_test, y_pred_reg, beta=2, average='macro', zero_division=0)
    
    ablation_results.append({
        'component': 'Regularization',
        'variant': name,
        'cv_f2': round(cv_f2, 4),
        'test_f2': round(test_f2, 4),
        'notes': f'gap={gap:.4f} {"⚠️ overfit" if gap > 0.1 else "✅"}'
    })
    
    print(f'[Ablation] {name:<35} | CV F2: {cv_f2:.4f}  | Test F2: {test_f2:.4f}  | Gap: {gap:.4f}')


### 10.3 Imbalance Handling Strategies (XGBoost)

**Cách xử lý imbalance ảnh hưởng đến cân bằng giữa các class.**

Kiểm tra:
- `None` (không xử lý imbalance)
- `SMOTE only` (không có class weight)
- `class_weight='balanced' only` (không SMOTE)
- `SMOTE + class_weight='balanced'` (kết hợp)

---

In [ ]:
# ── Ablation: Imbalance handling strategies ────────────────────────────────────
imbalance_strategies = [
    {
        'name': 'No SMOTE, No class_weight',
        'pipe': XGBClassifier(n_estimators=300, learning_rate=0.1, max_depth=6,
                              eval_metric='mlogloss', verbosity=0, 
                              random_state=RANDOM_STATE, n_jobs=-1)
    },
    {
        'name': 'SMOTE only (no class_weight)',
        'pipe': make_smote_pipe(XGBClassifier(n_estimators=300, learning_rate=0.1, max_depth=6,
                                              eval_metric='mlogloss', verbosity=0,
                                              random_state=RANDOM_STATE, n_jobs=-1))
    },
    {
        'name': 'class_weight="balanced" only (no SMOTE)',
        'pipe': ImbPipeline([
            ('smote', 'passthrough'),  # Dummy passthrough
            ('clf', XGBClassifier(n_estimators=300, learning_rate=0.1, max_depth=6,
                                  scale_pos_weight=1,  # Will be handled via class weight
                                  eval_metric='mlogloss', verbosity=0,
                                  random_state=RANDOM_STATE, n_jobs=-1))
        ])
    },
    {
        'name': 'SMOTE + class_weight="balanced"',
        'pipe': make_smote_pipe(XGBClassifier(n_estimators=300, learning_rate=0.1, max_depth=6,
                                              eval_metric='mlogloss', verbosity=0,
                                              random_state=RANDOM_STATE, n_jobs=-1))
    },
]

for strategy in imbalance_strategies:
    name = strategy['name']
    pipe = strategy['pipe']
    
    cv_res = cross_validate(pipe, X_train_pp, y_train, cv=skf_ablation,
                             scoring={'f2_macro': f2_macro_scorer},
                             return_train_score=True, n_jobs=-1)
    cv_f2 = cv_res['test_f2_macro'].mean()
    
    pipe.fit(X_train_pp, y_train)
    y_pred_strat = pipe.predict(X_test_pp)
    test_f2 = fbeta_score(y_test, y_pred_strat, beta=2, average='macro', zero_division=0)
    
    ablation_results.append({
        'component': 'Imbalance Handling',
        'variant': name,
        'cv_f2': round(cv_f2, 4),
        'test_f2': round(test_f2, 4),
        'notes': ''
    })
    
    print(f'[Ablation] {name:<40} | CV F2: {cv_f2:.4f}  | Test F2: {test_f2:.4f}')


### 10.4 Feature Engineering Impact

**Feature engineering (is_very_low_elev, rain_x_return) có ảnh hưởng bao nhiêu?**

Kiểm tra:
- Không feature engineering (chỉ raw features)
- Với feature engineering

---

In [ ]:
# ── Ablation: Feature Engineering Impact ────────────────────────────────────────
# Chuẩn bị features: without engineered features
CORE_FEATURES = [f for f in NUM_FEATURES if f not in ['is_very_low_elev', 'rain_x_return']] + CAT_FEATURES
X_train_core = X_train_pp[CORE_FEATURES].copy()
X_test_core = X_test_pp[CORE_FEATURES].copy()

print(f'Full feature set: {len(ALL_FEATURES)} features')
print(f'Core (no FE) set: {len(CORE_FEATURES)} features')

fe_configs = [
    {
        'name': 'Without Feature Engineering (core only)',
        'X_train': X_train_core,
        'X_test': X_test_core,
    },
    {
        'name': 'With Feature Engineering (all features)',
        'X_train': X_train_pp,
        'X_test': X_test_pp,
    },
]

for config in fe_configs:
    name = config['name']
    X_tr, X_te = config['X_train'], config['X_test']
    
    xgb_fe = make_smote_pipe(XGBClassifier(
        n_estimators=300, learning_rate=0.1, max_depth=6,
        eval_metric='mlogloss', verbosity=0,
        random_state=RANDOM_STATE, n_jobs=-1
    ))
    
    cv_res = cross_validate(xgb_fe, X_tr, y_train, cv=skf_ablation,
                             scoring={'f2_macro': f2_macro_scorer},
                             return_train_score=True, n_jobs=-1)
    cv_f2 = cv_res['test_f2_macro'].mean()
    
    xgb_fe.fit(X_tr, y_train)
    y_pred_fe = xgb_fe.predict(X_te)
    test_f2 = fbeta_score(y_test, y_pred_fe, beta=2, average='macro', zero_division=0)
    
    ablation_results.append({
        'component': 'Feature Engineering',
        'variant': name,
        'cv_f2': round(cv_f2, 4),
        'test_f2': round(test_f2, 4),
        'notes': f'Features: {len(X_tr.columns)}'
    })
    
    print(f'[Ablation] {name:<50} | CV F2: {cv_f2:.4f}  | Test F2: {test_f2:.4f}')


### 10.5 Preprocessing Impact

**Preprocessing (clip outliers) có ảnh hưởng bao nhiêu?**

Kiểm tra:
- Raw data (không clip)
- Với clip

---

In [ ]:
# ── Ablation: Preprocessing Impact ──────────────────────────────────────────────
pp_configs = [
    {
        'name': 'Raw (no clip, no preprocessing)',
        'X_train': X_train,
        'X_test': X_test,
    },
    {
        'name': 'Clipped outliers (current preprocessing)',
        'X_train': X_train_pp,
        'X_test': X_test_pp,
    },
]

for config in pp_configs:
    name = config['name']
    X_tr_pp, X_te_pp = config['X_train'][ALL_FEATURES], config['X_test'][ALL_FEATURES]
    
    xgb_pp = make_smote_pipe(XGBClassifier(
        n_estimators=300, learning_rate=0.1, max_depth=6,
        eval_metric='mlogloss', verbosity=0,
        random_state=RANDOM_STATE, n_jobs=-1
    ))
    
    cv_res = cross_validate(xgb_pp, X_tr_pp, y_train, cv=skf_ablation,
                             scoring={'f2_macro': f2_macro_scorer},
                             return_train_score=True, n_jobs=-1)
    cv_f2 = cv_res['test_f2_macro'].mean()
    
    xgb_pp.fit(X_tr_pp, y_train)
    y_pred_pp = xgb_pp.predict(X_te_pp)
    test_f2 = fbeta_score(y_test, y_pred_pp, beta=2, average='macro', zero_division=0)
    
    ablation_results.append({
        'component': 'Preprocessing',
        'variant': name,
        'cv_f2': round(cv_f2, 4),
        'test_f2': round(test_f2, 4),
        'notes': ''
    })
    
    print(f'[Ablation] {name:<50} | CV F2: {cv_f2:.4f}  | Test F2: {test_f2:.4f}')


### 10.6 Ablation Summary

---

In [ ]:
# ── Ablation Summary Table ──────────────────────────────────────────────────────
ablation_df = pd.DataFrame(ablation_results)
print('📊 ABLATION STUDIES SUMMARY')
print('='*110)
print(ablation_df.to_string(index=False))

# Export ablation results
ablation_df.to_csv('ablation_studies_results.csv', index=False)
print(f'\n✅ Saved: ablation_studies_results.csv')

In [ ]:
# ── Visualize Ablation: Component Comparison ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot by component group
components = ablation_df['component'].unique()
colors_map = {
    'Loss Function': '#FF6B6B',
    'Regularization': '#4ECDC4',
    'Imbalance Handling': '#45B7D1',
    'Feature Engineering': '#96CEB4',
    'Preprocessing': '#FFEAA7',
}

# Test F2 by component
ablation_sorted = ablation_df.sort_values('test_f2', ascending=True)
bar_colors = [colors_map.get(comp, '#999') for comp in ablation_sorted['component']]
bars = axes[0].barh(range(len(ablation_sorted)), ablation_sorted['test_f2'], color=bar_colors, edgecolor='white', lw=1)
axes[0].set_yticks(range(len(ablation_sorted)))
axes[0].set_yticklabels([f"{row['variant']}\n({row['component']})" 
                          for _, row in ablation_sorted.iterrows()], fontsize=8)
axes[0].set_xlabel('Test F2-macro', fontsize=10)
axes[0].set_title('[TEST SET] Ablation Results — Test F2-macro', fontsize=11, fontweight='bold')
axes[0].axvline(ablation_sorted['test_f2'].max(), color='red', ls='--', alpha=0.3, label='Best')
axes[0].axvline(ablation_sorted['test_f2'].min(), color='blue', ls='--', alpha=0.3, label='Worst')
axes[0].legend()

# CV vs Test comparison
ax_twin = axes[1]
x_pos = np.arange(len(ablation_df))
width = 0.35

bars1 = ax_twin.bar(x_pos - width/2, ablation_df['cv_f2'], width, label='CV F2', alpha=0.8)
bars2 = ax_twin.bar(x_pos + width/2, ablation_df['test_f2'], width, label='Test F2', alpha=0.8)

ax_twin.set_ylabel('F2-macro', fontsize=10)
ax_twin.set_xlabel('Variant Index', fontsize=10)
ax_twin.set_title('CV vs Test F2 — All Ablations', fontsize=11, fontweight='bold')
ax_twin.set_xticks(x_pos)
ax_twin.set_xticklabels([v[:15]+'...' if len(v) > 15 else v for v in ablation_df['variant']], 
                          rotation=45, ha='right', fontsize=7)
ax_twin.legend()

plt.suptitle('Ablation Studies Analysis — Urban Flood Risk', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Detailed Ablation Analysis ──────────────────────────────────────────────────
print('\n📈 ABLATION INSIGHTS — Impact per Component\n')
print('='*100)

for component in ablation_df['component'].unique():
    sub = ablation_df[ablation_df['component'] == component]
    
    print(f'\n{component.upper()}')
    print('-'*100)
    
    # Min/Max/Range
    cv_min, cv_max = sub['cv_f2'].min(), sub['cv_f2'].max()
    test_min, test_max = sub['test_f2'].min(), sub['test_f2'].max()
    
    print(f'  CV   F2: {cv_min:.4f} → {cv_max:.4f}   (Range: {cv_max-cv_min:+.4f})')
    print(f'  Test F2: {test_min:.4f} → {test_max:.4f}   (Range: {test_max-test_min:+.4f})')
    
    # Best variant
    best_row = sub.loc[sub['test_f2'].idxmax()]
    worst_row = sub.loc[sub['test_f2'].idxmin()]
    
    print(f'  ✅ Best  : {best_row["variant"]:<45} | Test F2: {best_row["test_f2"]:.4f}')
    print(f'  ❌ Worst : {worst_row["variant"]:<45} | Test F2: {worst_row["test_f2"]:.4f}')
    
    # Improvement
    improvement = (best_row['test_f2'] - worst_row['test_f2']) / worst_row['test_f2'] * 100
    print(f'  💡 Improvement: {improvement:+.1f}% (by changing {component})')

# Overall best
print(f'\n{"="*100}')
best_overall = ablation_df.loc[ablation_df['test_f2'].idxmax()]
print(f'🏆 OVERALL BEST (Ablation):')
print(f'   Variant: {best_overall["variant"]}')
print(f'   Component: {best_overall["component"]}')  
print(f'   CV F2: {best_overall["cv_f2"]:.4f}  |  Test F2: {best_overall["test_f2"]:.4f}')


### 10.7 Feature Engineering Groups Ablation

**Test impact của từng FE group: G1 (Spatial), G2 (Hydraulic), G3 (Soil), G4 (Extreme), G5 (Geo), G6 (Log).**

Strategie:
- Baseline: original features only (NUM_FEATURES[:7] + CAT)
- +G1, +G2, ... : thêm từng group
- All: tất cả

---

In [ ]:
# ── FE Group Ablation: Build datasets with progressively added groups ────────
# Baseline: original features (without engineered)
original_num_features = NUM_FEATURES[:7]  # exclude engineered G1-G6 (indices 0-6 are original)
baseline_features = original_num_features + CAT_FEATURES

# Define feature combinations
fe_ablation_configs = [
    {
        'name': 'Baseline (no engineered FE)',
        'features': baseline_features,
        'groups': 'none'
    },
    {
        'name': '+G1 (Spatial Risk Compound)',
        'features': baseline_features + FE_GROUPS['G1'],
        'groups': 'G1'
    },
    {
        'name': '+G2 (Hydraulic Load)',
        'features': baseline_features + FE_GROUPS['G2'],
        'groups': 'G2'
    },
    {
        'name': '+G3 (Soil × Rainfall)',
        'features': baseline_features + FE_GROUPS['G3'],
        'groups': 'G3'
    },
    {
        'name': '+G4 (Extreme Event)',
        'features': baseline_features + FE_GROUPS['G4'],
        'groups': 'G4'
    },
    {
        'name': '+G5 (Geographic)',
        'features': baseline_features + FE_GROUPS['G5'],
        'groups': 'G5'
    },
    {
        'name': '+G6 (Log Transform)',
        'features': baseline_features + FE_GROUPS['G6'],
        'groups': 'G6'
    },
    {
        'name': 'All FE Groups (G1+G2+G3+G4+G5+G6)',
        'features': ALL_FEATURES_EXTENDED,
        'groups': 'G1+G2+G3+G4+G5+G6'
    },
]

fe_ablation_log = []

for config in fe_ablation_configs:
    name = config['name']
    features = config['features']
    groups = config['groups']
    
    # Prepare data
    X_tr_fe = X_train[features].copy()
    X_te_fe = X_test[features].copy()
    
    # Train model
    xgb_fe = make_smote_pipe(XGBClassifier(
        n_estimators=300, learning_rate=0.1, max_depth=6,
        eval_metric='mlogloss', verbosity=0,
        random_state=RANDOM_STATE, n_jobs=-1
    ))
    
    # CV evaluation
    cv_res = cross_validate(xgb_fe, X_tr_fe, y_train, cv=skf_ablation,
                             scoring={'f2_macro': f2_macro_scorer},
                             return_train_score=True, n_jobs=-1)
    cv_f2 = cv_res['test_f2_macro'].mean()
    train_f2 = cv_res['train_f2_macro'].mean()
    gap = train_f2 - cv_f2
    
    # Test evaluation
    xgb_fe.fit(X_tr_fe, y_train)
    y_pred_fe = xgb_fe.predict(X_te_fe)
    test_f2 = fbeta_score(y_test, y_pred_fe, beta=2, average='macro', zero_division=0)
    test_f1w = f1_score(y_test, y_pred_fe, average='weighted', zero_division=0)
    
    fe_ablation_log.append({
        'component': 'Feature Engineering',
        'variant': name,
        'groups': groups,
        'num_features': len(features),
        'cv_f2': round(cv_f2, 4),
        'train_f2': round(train_f2, 4),
        'gap': round(gap, 4),
        'test_f2': round(test_f2, 4),
        'test_f1w': round(test_f1w, 4),
    })
    
    print(f'[FE Ablation] {name:<50} | Feat: {len(features):>3} | CV F2: {cv_f2:.4f} | Test F2: {test_f2:.4f}')
    
    # Add to main ablation log
    ablation_results.append({
        'component': 'Feature Engineering',
        'variant': name,
        'cv_f2': round(cv_f2, 4),
        'test_f2': round(test_f2, 4),
        'notes': f'Groups: {groups} | Features: {len(features)}'
    })

fe_ablation_df = pd.DataFrame(fe_ablation_log)
print(f'\n✅ FE Group Ablation Complete: {len(fe_ablation_df)} configurations tested')

In [ ]:
# ── FE Ablation Visualization ──────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Plot 1: Test F2 by FE config
ax=axes[0,0]
sorted_fe = fe_ablation_df.sort_values('test_f2', ascending=True)
colors_fe = ['#F44336' if 'baseline' in v.lower() else '#4CAF50' if 'all' in v.lower() else '#90CAF9'
             for v in sorted_fe['variant']]
ax.barh(range(len(sorted_fe)), sorted_fe['test_f2'], color=colors_fe, edgecolor='white', lw=1)
ax.set_yticks(range(len(sorted_fe)))
ax.set_yticklabels([v.replace('(', '\n(') for v in sorted_fe['variant']], fontsize=8)
ax.set_xlabel('Test F2-macro', fontsize=10)
ax.set_title('[TEST] FE Group Impact on F2-macro', fontsize=11, fontweight='bold')
ax.axvline(sorted_fe['test_f2'].max(), color='green', ls='--', alpha=0.3, label='Best')

# Plot 2: CV vs Test F2
ax=axes[0,1]
x_pos = np.arange(len(fe_ablation_df))
width = 0.35
ax.bar(x_pos - width/2, fe_ablation_df['cv_f2'], width, label='CV F2', alpha=0.8)
ax.bar(x_pos + width/2, fe_ablation_df['test_f2'], width, label='Test F2', alpha=0.8)
ax.set_ylabel('F2-macro', fontsize=10)
ax.set_title('CV vs Test F2 — All FE Configs', fontsize=11, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels([i for i in range(len(fe_ablation_df))], fontsize=9)
ax.legend()

# Plot 3: Number of features vs performance
ax=axes[1,0]
scatter = ax.scatter(fe_ablation_df['num_features'], fe_ablation_df['test_f2'], 
                     s=150, c=range(len(fe_ablation_df)), cmap='viridis', 
                     edgecolor='white', lw=1.5, alpha=0.8)
for idx, (x, y, v) in enumerate(zip(fe_ablation_df['num_features'], fe_ablation_df['test_f2'], fe_ablation_df['variant'])):
    label = v.split('(')[0].strip()[:15]  # Short label
    ax.annotate(label, (x, y), xytext=(5, 5), textcoords='offset points', fontsize=8)
ax.set_xlabel('Number of Features', fontsize=10)
ax.set_ylabel('Test F2-macro', fontsize=10)
ax.set_title('Feature Count vs Performance Trade-off', fontsize=11, fontweight='bold')
plt.colorbar(scatter, ax=ax, label='Config Index')

# Plot 4: Overfitting gap (train F2 - CV F2)
ax=axes[1,1]
sorted_gap = fe_ablation_df.sort_values('gap', ascending=False)
colors_gap = ['#F44336' if g > 0.1 else '#FFC107' if g > 0.05 else '#4CAF50' 
              for g in sorted_gap['gap']]
ax.barh(range(len(sorted_gap)), sorted_gap['gap'], color=colors_gap, edgecolor='white', lw=1)
ax.set_yticks(range(len(sorted_gap)))
ax.set_yticklabels(sorted_gap['groups'], fontsize=9)
ax.set_xlabel('Overfit Gap (Train F2 - CV F2)', fontsize=10)
ax.set_title('Overfitting Analysis by FE Group', fontsize=11, fontweight='bold')
ax.axvline(0.1, color='red', ls='--', alpha=0.4, linewidth=1, label='Overfit threshold')
ax.legend()

plt.suptitle('Feature Engineering Ablation Studies — Comprehensive Analysis', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── FE Ablation Summary & Insights ─────────────────────────────────────────────
print('\n📊 FEATURE ENGINEERING ABLATION — DETAILED ANALYSIS')
print('='*120)
print(fe_ablation_df.to_string(index=False))

print('\n\n💡 KEY INSIGHTS:')
print('-'*120)

# Baseline performance
baseline_row = fe_ablation_df[fe_ablation_df['groups'] == 'none'].iloc[0]
best_row = fe_ablation_df.loc[fe_ablation_df['test_f2'].idxmax()]
all_row = fe_ablation_df[fe_ablation_df['groups'] == 'G1+G2+G3+G4+G5+G6'].iloc[0]

print(f'\n1. BASELINE vs BEST:')
print(f'   Baseline (no FE)       : Test F2 = {baseline_row["test_f2"]:.4f}')
print(f'   Best single group      : {best_row["variant"]} = {best_row["test_f2"]:.4f}')
improvement_best = (best_row['test_f2'] - baseline_row['test_f2']) / baseline_row['test_f2'] * 100
print(f'   → Improvement: {improvement_best:+.1f}%')

print(f'\n2. BASELINE vs ALL FE:')
print(f'   Baseline (no FE)       : Test F2 = {baseline_row["test_f2"]:.4f}  ({baseline_row["num_features"]:>2} features)')
print(f'   All FE groups          : Test F2 = {all_row["test_f2"]:.4f}  ({all_row["num_features"]:>2} features)')
improvement_all = (all_row['test_f2'] - baseline_row['test_f2']) / baseline_row['test_f2'] * 100
print(f'   → Improvement: {improvement_all:+.1f}%')
print(f'   → Feature increase: {all_row["num_features"] - baseline_row["num_features"]} new features')

# Group-by-group impact
print(f'\n3. INDIVIDUAL GROUP CONTRIBUTION (vs Baseline):')
single_groups = fe_ablation_df[fe_ablation_df['groups'].str.len() <= 2]
for _, row in single_groups.iterrows():
    delta = (row['test_f2'] - baseline_row['test_f2']) / baseline_row['test_f2'] * 100
    overfit = '⚠️ overfit' if row['gap'] > 0.1 else '✅'
    print(f'   {row["groups"]:<5} : {row["test_f2"]:.4f}  ({delta:+.1f}%)  | Gap: {row["gap"]:.4f} {overfit}')

# Best trade-off (performance vs # features)
print(f'\n4. PERFORMANCE-FEATURES TRADE-OFF:')
best_efficiency = fe_ablation_df.loc[
    (fe_ablation_df['test_f2'] / fe_ablation_df['num_features']).idxmax()
]
print(f'   Best efficiency (F2/features):')
print(f'   {best_efficiency["variant"]}')
# Overfit analysis
print(f'\n5. OVERFITTING RISK:')
fe_ablation_df['overfit_risk'] = fe_ablation_df['gap'].apply(
    lambda x: '🔴 HIGH (>0.1)' if x > 0.1 else '🟡 MEDIUM (0.05-0.1)' if x > 0.05 else '🟢 LOW (<0.05)'
)
print(fe_ablation_df[['groups', 'test_f2', 'gap', 'overfit_risk']].to_string(index=False))

# Export result
fe_ablation_df.to_csv('fe_ablation_results.csv', index=False)
print(f'\n✅ Saved: fe_ablation_results.csv')


### 10.8 Comprehensive Ablation Summary

---

In [ ]:
# ── Combine all ablation results (including FE) ──────────────────────────────
ablation_df_complete = pd.DataFrame(ablation_results)

print('\n📊 COMPREHENSIVE ABLATION STUDIES — ALL COMPONENTS')
print('='*110)
print(ablation_df_complete.to_string(index=False))

# Summary by component
print('\n\n📈 ABLATION BREAKDOWN BY COMPONENT:')
print('='*110)
for component in ablation_df_complete['component'].unique():
    sub = ablation_df_complete[ablation_df_complete['component'] == component]
    test_f2_range = (sub['test_f2'].max() - sub['test_f2'].min())
    best_variant = sub.loc[sub['test_f2'].idxmax()]
    
    print(f'\n{component.upper()}:')
    print(f'  # Variants tested: {len(sub)}')
    print(f'  Test F2 range: {sub["test_f2"].min():.4f} → {sub["test_f2"].max():.4f}  (Δ = {test_f2_range:+.4f})')
    print(f'  Best variant: {best_variant["variant"]}')
    print(f'  → Test F2: {best_variant["test_f2"]:.4f}')

# Top 10 overall
print(f'\n\n🏆 TOP 10 VARIANTS (by Test F2):')
print('='*110)
top10 = ablation_df_complete.nlargest(10, 'test_f2')[['component', 'variant', 'test_f2', 'cv_f2']]
for idx, (_, row) in enumerate(top10.iterrows(), 1):
    print(f'{idx:2}. {row["component"]:<25} | {row["variant"]:<50} | Test F2: {row["test_f2"]:.4f}')

# Save complete ablation
ablation_df_complete.to_csv('complete_ablation_results.csv', index=False)
print(f'\n✅ Saved: complete_ablation_results.csv  ({len(ablation_df_complete)} total ablation configs)')

In [ ]:
# ── Final comprehensive visualization ──────────────────────────────────────────
fig = plt.figure(figsize=(18, 10))
gs = fig.add_gridspec(3, 2, hspace=0.3, wspace=0.3)

# Plot 1: All variants ranked by Test F2
ax1 = fig.add_subplot(gs[0, :])
sorted_all = ablation_df_complete.sort_values('test_f2', ascending=True)
colors_all = {
    'Loss Function': '#FF6B6B',
    'Regularization': '#4ECDC4',
    'Imbalance Handling': '#45B7D1',
    'Feature Engineering': '#96CEB4',
    'Preprocessing': '#FFEAA7',
}
bar_colors = [colors_all.get(comp, '#999') for comp in sorted_all['component']]
ax1.barh(range(len(sorted_all)), sorted_all['test_f2'], color=bar_colors, edgecolor='white', lw=0.8)
ax1.set_yticks(range(len(sorted_all)))
ax1.set_yticklabels([f"{v[:40]}..." if len(v) > 40 else v for v in sorted_all['variant']], fontsize=7)
ax1.set_xlabel('Test F2-macro', fontsize=11)
ax1.set_title('All Ablation Variants Ranked by Performance', fontsize=12, fontweight='bold')
ax1.axvline(sorted_all['test_f2'].max(), color='green', ls='--', alpha=0.3, lw=2)

# Plot 2: Component comparison (best in each)
ax2 = fig.add_subplot(gs[1, 0])
best_per_component = ablation_df_complete.loc[ablation_df_complete.groupby('component')['test_f2'].idxmax()]
comp_order = ['Feature Engineering', 'Regularization', 'Imbalance Handling', 'Loss Function', 'Preprocessing']
best_per_component_sorted = best_per_component.reindex([c for c in comp_order if c in best_per_component.index])
bars = ax2.bar(range(len(best_per_component_sorted)), best_per_component_sorted['test_f2'],
               color=[colors_all.get(c, '#999') for c in best_per_component_sorted['component']],
               edgecolor='white', lw=1.5)
ax2.set_xticks(range(len(best_per_component_sorted)))
ax2.set_xticklabels([c.replace(' ', '\n') for c in best_per_component_sorted['component']], fontsize=9)
ax2.set_ylabel('Best Test F2-macro', fontsize=11)
ax2.set_title('Best Configuration per Component', fontsize=12, fontweight='bold')
for bar, v in zip(bars, best_per_component_sorted['test_f2']):
    ax2.text(bar.get_x() + bar.get_width()/2, v + 0.002, f'{v:.4f}', 
             ha='center', va='bottom', fontsize=9, fontweight='bold')

# Plot 3: Component impact (max - min)
ax3 = fig.add_subplot(gs[1, 1])
component_impact = ablation_df_complete.groupby('component')['test_f2'].agg(['min','max']).reset_index()
component_impact['impact'] = component_impact['max'] - component_impact['min']
component_impact = component_impact.sort_values('impact', ascending=True)
bars = ax3.barh(range(len(component_impact)), component_impact['impact'],
                color=[colors_all.get(c, '#999') for c in component_impact['component']],
                edgecolor='white', lw=1.5)
ax3.set_yticks(range(len(component_impact)))
ax3.set_yticklabels(component_impact['component'], fontsize=10)
ax3.set_xlabel('Impact (max F2 - min F2)', fontsize=11)
ax3.set_title('Component Sensitivity Analysis', fontsize=12, fontweight='bold')
for bar, v in zip(bars, component_impact['impact']):
    ax3.text(v + 0.002, bar.get_y() + bar.get_height()/2, f'{v:.4f}', 
             va='center', fontsize=9)

# Plot 4: Component distribution (all variants)
ax4 = fig.add_subplot(gs[2, :])
for component in comp_order:
    if component in ablation_df_complete['component'].values:
        sub = ablation_df_complete[ablation_df_complete['component'] == component]
        ax4.scatter([component] * len(sub), sub['test_f2'], 
                   s=100, alpha=0.6, label=component,
                   color=colors_all.get(component, '#999'))
ax4.set_ylabel('Test F2-macro', fontsize=11)
ax4.set_title('Distribution of All Variants per Component', fontsize=12, fontweight='bold')
ax4.grid(axis='y', alpha=0.3)

plt.suptitle('Comprehensive Ablation Studies — 5 Components × Multiple Variants', 
             fontsize=14, fontweight='bold', y=0.995)
plt.show()

In [ ]:
# ── Ablation Recommendations & Best Configuration ─────────────────────────────
print('\n\n🎯 ABLATION STUDIES — RECOMMENDATIONS')
print('='*110)

overall_best = ablation_df_complete.loc[ablation_df_complete['test_f2'].idxmax()]
print(f'\n🏆 BEST OVERALL CONFIGURATION:')
print(f'   Component: {overall_best["component"]}')
print(f'   Variant: {overall_best["variant"]}')
print(f'   Test F2-macro: {overall_best["test_f2"]:.4f}')

# Recommendations by component
print(f'\n\n📋 COMPONENT-WISE RECOMMENDATIONS:')
print('='*110)
for component in comp_order:
    if component in ablation_df_complete['component'].values:
        sub = ablation_df_complete[ablation_df_complete['component'] == component]
        best = sub.loc[sub['test_f2'].idxmax()]
        worst = sub.loc[sub['test_f2'].idxmin()]
        impact = best['test_f2'] - worst['test_f2']
        
        print(f'\n{component}:')
        print(f'  Best  : {best["variant"]:<50} | F2: {best["test_f2"]:.4f}')
        print(f'  Worst : {worst["variant"]:<50} | F2: {worst["test_f2"]:.4f}')
        print(f'  Impact: {impact:.4f} ({impact/worst["test_f2"]*100:+.1f}%)')

# Create recommendation summary
print(f'\n\n💡 KEY FINDINGS:')
print('='*110)

# Feature Engineering impact
fe_baselines = ablation_df_complete[ablation_df_complete['component'] == 'Feature Engineering']
fe_best = fe_baselines[fe_baselines['groups'] == 'G1+G2+G3+G4+G5+G6'].iloc[0] if any(fe_baselines['component'] == 'Feature Engineering') else None
fe_none = fe_baselines[fe_baselines['groups'] == 'none'].iloc[0] if any(fe_baselines['groups'] == 'none') else None

if fe_best is not None and fe_none is not None:
    fe_improvement = (fe_best['test_f2'] - fe_none['test_f2']) / fe_none['test_f2'] * 100
    print(f'\n1. FEATURE ENGINEERING IMPACT:')
    print(f'   All FE groups contribute {fe_improvement:+.1f}% improvement')
    print(f'   → {fe_best["num_features"]} total features (vs {fe_none["num_features"]} baseline)')
    print(f'   → Recommendation: Use ALL FE groups for best performance')

# Regularization impact
reg_sub = ablation_df_complete[ablation_df_complete['component'] == 'Regularization']
if len(reg_sub) > 0:
    reg_best = reg_sub.loc[reg_sub['test_f2'].idxmax()]
    reg_worst = reg_sub.loc[reg_sub['test_f2'].idxmin()]
    print(f'\n2. REGULARIZATION IMPACT:')
    print(f'   Best regularization: {reg_best["variant"]} ({reg_best["test_f2"]:.4f})')
    print(f'   Penalty for no regularization: {(reg_worst["test_f2"]-reg_best["test_f2"])*100:.2f}%')

# Imbalance handling impact
imb_sub = ablation_df_complete[ablation_df_complete['component'] == 'Imbalance Handling']
if len(imb_sub) > 0:
    imb_best = imb_sub.loc[imb_sub['test_f2'].idxmax()]
    imb_none = imb_sub[imb_sub['variant'].str.contains('No SMOTE')] if any(imb_sub['variant'].str.contains('No SMOTE')) else None
    if imb_none is not None and len(imb_none) > 0:
        imb_improvement = (imb_best['test_f2'] - imb_none.iloc[0]['test_f2']) / imb_none.iloc[0]['test_f2'] * 100
        print(f'\n3. IMBALANCE HANDLING IMPACT:')
        print(f'   Best strategy: {imb_best["variant"]} ({imb_best["test_f2"]:.4f})')
        print(f'   Improvement vs no handling: {imb_improvement:+.1f}%')

print(f'\n{"="*110}')
print(f'✅ Ablation studies complete! Refer to:')
print(f'   - ablation_studies_results.csv (original ablations)')
print(f'   - fe_ablation_results.csv (feature engineering)')
print(f'   - complete_ablation_results.csv (all combined)')
